# Дообучение `intfloat/multilingual-e5-large` на банковских документах

**Стратегия:** консервативное дообучение с частичной разморозкой
- Этап 1: TSDAE — SimCSE-MNRL domain adaptation (без разметки)
- Этап 2: MNRL — supervised fine-tuning на стратегии `adjacent`

**Ключевые отличия от базового пайплайна:**

| Параметр | USER-bge-m3 (base) | multilingual-e5-large |
|---|---|---|
| Backbone | XLM-RoBERTa-base | XLM-RoBERTa-large |
| Слоёв encoder | 12 | 24 |
| Параметры | ~278M | ~560M |
| `unfreeze_last_n_layers` | 5 | **8** |
| `learning_rate` | 2e-5 | **1e-5** |
| `warmup_ratio` | 0.1 | **0.2** |
| Стратегия позитивов | all | **adjacent only** |
| Инференс: префиксы | нет | **`query:` / `passage:`** |


## 0. Окружение и параметры

In [ ]:
!nvidia-smi 2>/dev/null || echo "CPU only"

In [ ]:
import sys, os, warnings, logging
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

from src import EmbeddingTrainer, LossType
from src.trainer import make_cpu_config, make_gpu_config
from src.data_preparation import Chunk, TripletSampler
from src.metrics import RetrievalEvaluator, build_evaluation_set

DATA  = Path("../data")
OUT   = Path("../output/e5_large");  OUT.mkdir(parents=True, exist_ok=True)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "intfloat/multilingual-e5-large"

# ═══════════════════════════════════════════════════════════════════════════════
# ПАРАМЕТРЫ — все изменения делай здесь
# ═══════════════════════════════════════════════════════════════════════════════
NUM_EPOCHS_TSDAE      = 10
NUM_EPOCHS_MNRL       = 10
EARLY_STOPPING        = 2
LOG_EVERY             = "epoch"

# Layer-wise fine-tuning для 24-слойной модели
UNFREEZE_LAST_N_LAYERS = 8    # ~13% параметров
DROPOUT_LAST_M_LAYERS  = 4
DROPOUT_P              = 0.2  # усиленная регуляризация

# Данные
N_PER_CHUNK    = 1            # меньше, но чище
MAX_CHUNKS     = 3_000
MAX_CHUNKS_QT  = None         # query_template отключён (только adjacent)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"Устройство  : {DEVICE}")
print(f"PyTorch     : {torch.__version__}")
print(f"Модель      : {MODEL_NAME}")
if DEVICE == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"\nПараметры  :")
print(f"  unfreeze_last_n_layers = {UNFREEZE_LAST_N_LAYERS}")
print(f"  dropout p={DROPOUT_P} в последних {DROPOUT_LAST_M_LAYERS} слоях")
print(f"  learning_rate          = 1e-5")
print(f"  warmup_ratio           = 0.2")
print(f"  стратегия позитивов    = adjacent only")

## 1. Загрузка корпуса

Корпус генерируется в ноутбуке `finetune_banking_embeddings.ipynb`, Секция 0.

In [ ]:
corpus_df = pd.read_csv(DATA / "stage1_tsdae" / "corpus.csv")
print(f"Чанков    : {len(corpus_df)}")
print(f"Документов: {corpus_df['doc_id'].nunique()}")
corpus_df.head(3)

## 2. Этап 1 — TSDAE (доменная адаптация)

Обучаем модель «узнавать» зашумлённый банковский текст через SimCSE-MNRL на парах
`(noised_sentence, original_sentence)`.

Параметры специально занижены по сравнению с базовым пайплайном:
- `learning_rate = 1e-5` — e5-large больше, легче переобучить
- `unfreeze_last_n_layers = 8` — из 24 слоёв, ~13% весов
- `warmup_ratio = 0.2` — плавный старт для большой модели


In [ ]:
stage1_df = corpus_df.rename(columns={"sentence": "anchor"}) if "anchor" not in corpus_df.columns     else corpus_df
# TripletSampler ожидает колонку sentence — TSDAE trainer берёт её напрямую
stage1_df = corpus_df.copy()

if DEVICE == "cpu":
    cfg_tsdae = make_cpu_config(
        loss_type=LossType.TSDAE,
        num_epochs=NUM_EPOCHS_TSDAE,
        output_dir=str(OUT / "stage1_tsdae"),
    )
else:
    cfg_tsdae = make_gpu_config(
        loss_type=LossType.TSDAE,
        num_epochs=NUM_EPOCHS_TSDAE,
        output_dir=str(OUT / "stage1_tsdae"),
    )

# e5-large специфичные гиперпараметры
cfg_tsdae.model_name               = MODEL_NAME
cfg_tsdae.learning_rate            = 1e-5
cfg_tsdae.warmup_ratio             = 0.2
cfg_tsdae.tsdae_noise_ratio        = 0.6
cfg_tsdae.early_stopping_patience  = EARLY_STOPPING
cfg_tsdae.logging_strategy         = LOG_EVERY
cfg_tsdae.unfreeze_last_n_layers   = UNFREEZE_LAST_N_LAYERS
cfg_tsdae.dropout_last_m_layers    = DROPOUT_LAST_M_LAYERS
cfg_tsdae.dropout_p                = DROPOUT_P

print("Конфигурация TSDAE:")
for k in ["model_name", "loss_type", "learning_rate", "warmup_ratio",
          "num_epochs", "batch_size", "unfreeze_last_n_layers",
          "dropout_last_m_layers", "dropout_p"]:
    print(f"  {k:30s}: {getattr(cfg_tsdae, k)}")

In [ ]:
trainer_tsdae = EmbeddingTrainer(cfg_tsdae)
trainer_tsdae.train(stage1_df)

## 3. Этап 2 — MNRL (только стратегия `adjacent`)

Используем самые надёжные позитивы — соседние чанки одного документа.
Отказываемся от `query_template`, чтобы снизить шум в малом корпусе.

**Почему `adjacent` лучше на малых данных:**
Шаблонные запросы (`query_template`) нередко не совпадают по стилю с реальными
вопросами пользователей, добавляя ложные сигналы. `adjacent` — детерминированный
позитив, который гарантированно тематически близок к якорю.


In [ ]:
# Загружаем корпус и строим чанки (section=None — корпус без секций)
chunks = [
    Chunk(text=row["sentence"], doc_id=row["doc_id"],
          chunk_id=row["chunk_id"], section=None)
    for row in corpus_df.to_dict("records")
]
print(f"Чанков: {len(chunks)} | Документов: {len(set(c.doc_id for c in chunks))}")

sampler = TripletSampler(chunks, negative_strategy="bm25", seed=42,
                         max_chunks=MAX_CHUNKS)
all_df = sampler.to_dataframe(
    strategies=["adjacent"],
    n_per_chunk=N_PER_CHUNK,
)

train_df, eval_df = train_test_split(all_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)

print(f"Триплетов: {len(all_df)}  | Train: {len(train_df)}  | Eval: {len(eval_df)}")

In [ ]:
tsdae_model_path = str(OUT / "stage1_tsdae" / "best_unsupervised")

if DEVICE == "cpu":
    cfg_mnrl = make_cpu_config(
        loss_type=LossType.MNRL,
        num_epochs=NUM_EPOCHS_MNRL,
        output_dir=str(OUT / "stage2_mnrl"),
    )
else:
    cfg_mnrl = make_gpu_config(
        loss_type=LossType.MNRL,
        num_epochs=NUM_EPOCHS_MNRL,
        output_dir=str(OUT / "stage2_mnrl"),
    )

cfg_mnrl.model_name              = tsdae_model_path   # стартуем с TSDAE-чекпоинта
cfg_mnrl.learning_rate           = 1e-5
cfg_mnrl.warmup_ratio            = 0.2
cfg_mnrl.early_stopping_patience = EARLY_STOPPING
cfg_mnrl.logging_strategy        = LOG_EVERY
cfg_mnrl.unfreeze_last_n_layers  = UNFREEZE_LAST_N_LAYERS
cfg_mnrl.dropout_last_m_layers   = DROPOUT_LAST_M_LAYERS
cfg_mnrl.dropout_p               = DROPOUT_P

print("Конфигурация MNRL:")
for k in ["model_name", "loss_type", "learning_rate", "warmup_ratio",
          "num_epochs", "batch_size", "unfreeze_last_n_layers"]:
    print(f"  {k:30s}: {getattr(cfg_mnrl, k)}")

In [ ]:
trainer_mnrl = EmbeddingTrainer(cfg_mnrl)
trainer_mnrl.train(train_df, eval_df)

## 4. Оценка качества

**Важно:** multilingual-e5-large требует префиксов при инференсе:
- Запросы → `"query: {текст}"`
- Документы (корпус) → `"passage: {текст}"`

Без префиксов модель работает, но теряет ~5–10 Recall@k.
Реализуем через `prompt` параметр SentenceTransformer.


In [ ]:
from sentence_transformers import SentenceTransformer
from src.metrics import RetrievalEvaluator

def evaluate_e5(model_path_or_name: str, label: str, use_prefix: bool = True):
    """Оценивает модель с учётом e5-префиксов."""
    model = SentenceTransformer(model_path_or_name, device=DEVICE)

    # Строим корпус из eval-триплетов
    all_triplets = pd.concat([train_df, eval_df], ignore_index=True)
    corpus_texts = list(set(all_triplets["positive"].tolist() +
                            all_triplets["negative"].tolist()))

    if use_prefix:
        query_texts   = ["query: "   + q for q in all_triplets["anchor"].tolist()]
        corpus_pref   = ["passage: " + p for p in corpus_texts]
    else:
        query_texts   = all_triplets["anchor"].tolist()
        corpus_pref   = corpus_texts

    # Эмбеддинги
    q_emb  = model.encode(query_texts,  batch_size=64, show_progress_bar=False,
                          normalize_embeddings=True, convert_to_numpy=True)
    c_emb  = model.encode(corpus_pref,  batch_size=64, show_progress_bar=False,
                          normalize_embeddings=True, convert_to_numpy=True)

    # Простой Recall@k через матричное умножение
    import numpy as np
    sims = q_emb @ c_emb.T   # (n_queries, n_corpus)

    # Для каждого запроса ищем его ground-truth позитив
    pos_texts = (["query: " + t for t in all_triplets["positive"].tolist()]
                 if use_prefix else all_triplets["positive"].tolist())
    corpus_idx = {t: i for i, t in enumerate(corpus_pref)}

    results = {}
    for k in [1, 3, 5, 10]:
        hits = 0
        for i, pos in enumerate(pos_texts):
            gt_idx = corpus_idx.get(pos)
            if gt_idx is None:
                continue
            top_k = np.argpartition(sims[i], -k)[-k:]
            if gt_idx in top_k:
                hits += 1
        results[f"Recall@{k}"] = round(hits / len(pos_texts), 4)

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")
    return results

# Baseline
r_base = evaluate_e5(MODEL_NAME, "Baseline (без дообучения)")

# После TSDAE
r_tsdae = evaluate_e5(str(OUT / "stage1_tsdae" / "best_unsupervised"), "TSDAE")

# После MNRL
r_mnrl = evaluate_e5(str(OUT / "stage2_mnrl" / "best_mnrl"), "TSDAE → MNRL")

In [ ]:
# Сводная таблица
metrics = ["Recall@1", "Recall@3", "Recall@5", "Recall@10"]
summary = pd.DataFrame({
    "Baseline":    [r_base.get(m, 0)  for m in metrics],
    "TSDAE":       [r_tsdae.get(m, 0) for m in metrics],
    "TSDAE→MNRL":  [r_mnrl.get(m, 0)  for m in metrics],
}, index=metrics)

print(summary.to_string())

# Визуализация
fig, ax = plt.subplots(figsize=(8, 4))
summary.T.plot(kind="bar", ax=ax, width=0.6)
ax.set_title("multilingual-e5-large: Recall@k по этапам")
ax.set_xlabel("Модель")
ax.set_ylabel("Recall@k")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "recall_comparison.png", dpi=120)
plt.show()